In [2]:
import os
import zipfile
import numpy as np
from huggingface_hub import hf_hub_download, login
from pathlib import Path

BASE_DIR = Path("./comma2k19_data")
RAW_DATA_DIR = BASE_DIR / "raw_data"
EXTRACT_DIR = BASE_DIR / "extracted"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directories configured at: {BASE_DIR.resolve()}")

c:\Users\lokesh\assets\envs\physengine\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data directories configured at: C:\Users\lokesh\AllFiles\UCSD\ML-for-Physical-Applications\FSD_Project\comma2k19_data


In [ ]:
hf_token = None

print("Downloading Chunk.zip.")
file_path = hf_hub_download(
    repo_id="commaai/comma2k19",
    filename="raw_data/Chunk_1.zip",    # update for different datachunk (1-10)
    repo_type="dataset",
    local_dir=BASE_DIR,
    local_dir_use_symlinks=False,
    token=hf_token
)

# file checks
size_gb = os.path.getsize(file_path) / (1024**3)
print(f"File verified! Size: {size_gb:.2f} GB")
if size_gb < 0.1:
    print("Error: File is too small.")
else:
    print("Success! Data downloaded.")

c:\Users\lokesh\assets\envs\physengine\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


File verified! Size: 8.13 GB
Success! Data downloaded.


In [3]:
from tqdm import tqdm 

zip_path = RAW_DATA_DIR / "Chunk_1.zip" #update if you change chunk

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    files_to_extract = [m for m in all_files if not m.endswith('/')]
    print(f"Found {len(files_to_extract)} files in Chunk. Starting extraction...")

    for member in tqdm(files_to_extract, desc="Extracting Chunk"):
        clean_name = member.replace("|", "_")
        target_path = EXTRACT_DIR / clean_name
        target_path.parent.mkdir(parents=True, exist_ok=True)
        
        with zip_ref.open(member) as source, open(target_path, "wb") as target:
            target.write(source.read())

print(f"\nExtraction complete! Files located at: {EXTRACT_DIR.resolve()}")

Found 7333 files in Chunk. Starting extraction...


Extracting Chunk: 100%|██████████| 7333/7333 [00:45<00:00, 162.56it/s]


Extraction complete! Files located at: C:\Users\lokesh\AllFiles\UCSD\ML-for-Physical-Applications\FSD_Project\comma2k19_data\extracted


In [7]:
def load_comma_log(file_path):
    """Safely loads comma2k19 raw float64 logs."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")
    try:
        return np.load(file_path)
    except Exception:
        return np.fromfile(file_path, dtype=np.float64)

chunk_1_path = EXTRACT_DIR / "Chunk_1" # update 

#quick data check
for drive in chunk_1_path.iterdir():
    if drive.is_dir():
        print(f"--- Processing Folder: {drive.name} ---")

        segments = sorted([s for s in drive.iterdir() if s.is_dir()], key=lambda x: int(x.name) if x.name.isdigit() else x.name)
        
        for segment in segments:
            steering_path = segment / "processed_log" / "CAN" / "steering_angle" / "value"
            speed_path = segment / "processed_log" / "CAN" / "speed" / "value"
            
            try:
                steering_data = load_comma_log(steering_path)
                speed_data = load_comma_log(speed_path)
                
                print(f"  Segment {segment.name:2}: "
                      f"Steer [{np.min(steering_data):6.2f}, {np.max(steering_data):6.2f}] | "
                      f"Speed [{np.min(speed_data):6.2f}, {np.max(speed_data):6.2f}] | "
                      f"Data points [Speed = {len(speed_data)}, Steer = {len(steering_data)}]")
            
            except FileNotFoundError:
                print(f"  Segment {segment.name:2}: [!] Missing log files")
            except Exception as e:
                print(f"  Segment {segment.name:2}: [!] Error: {e}")
        print("\n")

--- Processing Folder: b0c9d2329ad1606b_2018-07-27--06-03-57 ---
  Segment 3 : Steer [-22.00,  12.30] | Speed [  1.30,  22.52] | Data points [Speed = 4974, Steer = 4973]
  Segment 4 : Steer [ -5.00,   6.60] | Speed [  0.00,  17.60] | Data points [Speed = 4974, Steer = 4975]
  Segment 5 : Steer [ -8.20,  14.30] | Speed [  0.00,  25.50] | Data points [Speed = 4973, Steer = 4972]
  Segment 6 : Steer [-10.20,   6.80] | Speed [ 23.29,  32.16] | Data points [Speed = 4973, Steer = 4973]
  Segment 7 : Steer [-11.80,  14.00] | Speed [ 22.10,  30.83] | Data points [Speed = 4973, Steer = 4973]
  Segment 8 : Steer [ -9.20,   9.90] | Speed [ 27.74,  32.06] | Data points [Speed = 4973, Steer = 4974]
  Segment 9 : Steer [-11.80,   9.50] | Speed [ 28.33,  33.66] | Data points [Speed = 4973, Steer = 4973]
  Segment 10: Steer [-11.70,   9.80] | Speed [ 30.43,  33.50] | Data points [Speed = 4973, Steer = 4973]
  Segment 11: Steer [-11.60,  24.20] | Speed [ 22.14,  31.61] | Data points [Speed = 4974, Stee

In [11]:
for drive in chunk_1_path.iterdir():
    if drive.is_dir():
        print(f"Verify Sampling: {drive.name}")

        segments = sorted([s for s in drive.iterdir() if s.is_dir()], key=lambda x: int(x.name) if x.name.isdigit() else x.name)
        
        for segment in segments:
            steer_t_path = segment / "processed_log" / "CAN" / "steering_angle" / "t"
            speed_t_path = segment / "processed_log" / "CAN" / "speed" / "t"
            
            try:
                s_t = load_comma_log(steer_t_path)
                v_t = load_comma_log(speed_t_path)
                
                # time diff
                s_diff = np.diff(s_t)
                v_diff = np.diff(v_t)

                s_mean = np.mean(s_diff)
                s_freq = 1.0 / s_mean if s_mean > 0 else 0
                s_jitter = np.std(s_diff)
                v_mean = np.mean(v_diff)
                v_freq = 1.0 / v_mean if v_mean > 0 else 0
                v_jitter = np.std(v_diff)

                print(f"  Seg {segment.name:2} | "
                      f"Steer: {s_freq:5.1f}Hz (avg {s_mean:.4f}s, ±{s_jitter:.5f}) | "
                      f"Speed: {v_freq:5.1f}Hz (avg {v_mean:.4f}s, ±{v_jitter:.5f})")
            
            except FileNotFoundError:
                print(f"  Segment {segment.name:2}: [!] Missing log files")
            except Exception as e:
                print(f"  Segment {segment.name:2}: [!] Error: {e}")
        print("\n")

Verify Sampling: b0c9d2329ad1606b_2018-07-27--06-03-57
  Seg 3  | Steer:  82.9Hz (avg 0.0121s, ±0.00408) | Speed:  82.9Hz (avg 0.0121s, ±0.00397)
  Seg 4  | Steer:  82.9Hz (avg 0.0121s, ±0.00414) | Speed:  82.9Hz (avg 0.0121s, ±0.00412)
  Seg 5  | Steer:  82.9Hz (avg 0.0121s, ±0.00430) | Speed:  82.9Hz (avg 0.0121s, ±0.00420)
  Seg 6  | Steer:  82.9Hz (avg 0.0121s, ±0.00411) | Speed:  82.9Hz (avg 0.0121s, ±0.00408)
  Seg 7  | Steer:  82.9Hz (avg 0.0121s, ±0.00420) | Speed:  82.9Hz (avg 0.0121s, ±0.00408)
  Seg 8  | Steer:  82.9Hz (avg 0.0121s, ±0.00414) | Speed:  82.9Hz (avg 0.0121s, ±0.00410)
  Seg 9  | Steer:  82.9Hz (avg 0.0121s, ±0.00415) | Speed:  82.9Hz (avg 0.0121s, ±0.00419)
  Seg 10 | Steer:  82.9Hz (avg 0.0121s, ±0.00417) | Speed:  82.9Hz (avg 0.0121s, ±0.00418)
  Seg 11 | Steer:  82.9Hz (avg 0.0121s, ±0.00415) | Speed:  82.9Hz (avg 0.0121s, ±0.00410)


Verify Sampling: b0c9d2329ad1606b_2018-07-27--06-50-48
  Seg 7  | Steer:  82.9Hz (avg 0.0121s, ±0.00412) | Speed:  82.9Hz (a